# H20 匹配预测和持有期限实验

固定一个 H20 候选：沿用 25 个价格特征、成熟年度 prior/Ridge 选择和 benchmark-terminal 目标，预测、规划及持有期限均为 20 个交易日。固定 55 次拟合、8 个原生回放、10/25bp 两个费用场景；SOXX 为主要基准。

H5 经济参考未核验，新的报价时钟和独立窗口初始化同时改变，不能声称只识别期限的因果效果。历史已复用；不直接晋升。所有参数和门槛见 `preregister_plan.json` 与 `execution_plan.json`。

技术恢复：首次 native 检查拒绝仅 HOLD 未执行列中的约 -1.7e-16 舍入；保留失败 Recorder，使用同一 55 个拟合对象恢复，新增拟合数为零。模型、参数、原始持仓路径和 native 审计门槛未改变。见 recovery_plan.json。


In [1]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import sys, json
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "SPEC.md").exists())
sys.path.insert(0, str(root))
import qlib
from qlib.data import D
from qlib.workflow import R
card = root / "research" / "h20_matched_forecast_execution_20261010"
plan = json.loads((card / "execution_plan.json").read_text())
qlib.init(provider_uri=str(root / plan["source"]["providers"]["published"]), region="us", kernels=1)
prefit = json.loads((card / "prefit_summary.json").read_text())
print("Prefit native DatasetH:", prefit["native_training_dataset"])


2026/10/11 15:01:03 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[64826:MainThread](2026-10-11 15:01:03,253) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[64826:MainThread](2026-10-11 15:01:03,254) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[64826:MainThread](2026-10-11 15:01:03,254) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


Prefit native DatasetH: {'class_name': 'DatasetH', 'rows': 3956, 'features': 25, 'labels': ['soxl_return_h20', 'soxx_return_h20'], 'first_signal': '2011-01-03', 'terminal_unknown': 21, 'wrong_horizon_blocked': True, 'calendar_bound': True, 'prepared_sha256': 'df4285156f2a44063360eafd6124beb0639ac9688ef08ce9af84878972a50642'}


In [2]:
from src.workflow.h20_matched_experiment import run_h20_matched_experiment
plan_path = card / "recovery_plan.json"
attempt = card / "replay_recovery_attempt.json"
if attempt.exists():
    state = json.loads(attempt.read_text())
    if state["status"] != "complete":
        raise RuntimeError("Existing incomplete attempt: inspect its Recorder; do not refit")
    summary = json.loads((card / "summary.json").read_text())
    from src.core.recorder_sources import load_verified_recorder_objects
    objects, audit = load_verified_recorder_objects(card, expected_recorder_id=summary["recorder_id"], artifacts=["aggregate/final_summary.pkl"])
    assert objects["aggregate/final_summary.pkl"] == summary
else:
    summary = run_h20_matched_experiment(root=root, card=card, execution_plan_path=plan_path)
print(json.dumps({k: summary[k] for k in ("recorder_id", "fit_calls", "native_cases", "historical_gate_pass", "diagnostic_gate_pass", "route_decision")}, ensure_ascii=False, indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "recorder_id": "72455b5609ab4cc5b37808d7e10b2cb0",
  "fit_calls": 55,
  "native_cases": 8,
  "historical_gate_pass": false,
  "diagnostic_gate_pass": false,
  "route_decision": "end_h20_route_no_further_search"
}


In [3]:
import numpy as np, pandas as pd
from src.core.recorder_sources import load_verified_recorder_objects
objects, source_audit = load_verified_recorder_objects(card, expected_recorder_id=summary["recorder_id"], artifacts=["aggregate/case_results.pkl"])
cases = objects["aggregate/case_results.pkl"]
rows = []
for window in ("development", "reused_diagnostic"):
    table = cases[window + "::h20::10"]["forecast"]
    pd.testing.assert_frame_equal(table, cases[window + "::h20::25"]["forecast"], check_exact=True)
    mature = pd.to_datetime(table.label_available_at, utc=True) < pd.Timestamp("2026-09-25 21:00", tz="UTC")
    for asset in ("soxl", "soxx"):
        actual = table.loc[mature, "actual_" + asset]
        predicted = table.loc[mature, "predicted_" + asset]
        prior = table.loc[mature, "prior_" + asset]
        assert np.isfinite(np.c_[actual, predicted, prior]).all()
        mse, prior_mse = float(((predicted-actual)**2).mean()), float(((prior-actual)**2).mean())
        rows.append(dict(window=window, asset=asset, mature_rows=len(actual), horizon_sessions=20,
            prediction_mse=mse, prior_mse=prior_mse, mse_ratio=mse/prior_mse,
            rank_ic=float(predicted.rank().corr(actual.rank())),
            prior_rank_ic=float(prior.rank().corr(actual.rank()))))
information = pd.DataFrame(rows)
# A separate diagnostic run: never resume/relabel the completed investment-evidence run.
with R.start(experiment_name=summary["experiment"] + "_forecast_diagnostic", uri=str(card / "qlib_recorder")):
    diagnostic = R.get_recorder()
    diagnostic.log_params(parent_recorder_id=summary["recorder_id"], models_trained=0, scope="reused_history_diagnostic_only")
    diagnostic.save_objects(**{"forecast_information.pkl": information, "parent_source_audit.pkl": source_audit})
information.to_csv(card / "forecast_information.csv", index=False)
analysis = dict(parent_recorder_id=summary["recorder_id"], diagnostic_recorder_id=diagnostic.id,
    diagnostic_experiment_id=str(diagnostic.experiment_id), recorder_uri=str(card / "qlib_recorder"),
    models_trained=0, historical_alpha_proven=False, conclusion="Both assets/windows have MSE above their own historical-mean prior; point IC is diagnostic, without a confidence claim.")
(card / "analysis_summary.json").write_text(json.dumps(analysis, indent=2) + "\n")
display(information)


[64826:MainThread](2026-10-11 15:01:05,224) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name h20_matched_forecast_execution_20261010_forecast_diagnostic.


[64826:MainThread](2026-10-11 15:01:05,226) INFO - qlib.workflow - [exp.py:258] - Experiment 620007998418891224 starts running ...


[64826:MainThread](2026-10-11 15:01:05,311) INFO - qlib.workflow - [recorder.py:345] - Recorder 3556aa4c0f424e4f9d05070869b2e839 starts running under Experiment 620007998418891224 ...


[64826:MainThread](2026-10-11 15:01:06,239) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


,window,asset,mature_rows,horizon_sessions,prediction_mse,prior_mse,mse_ratio,rank_ic,prior_rank_ic
0,development,soxl,1762,20,0.055772,0.054873,1.016389,-0.113973,-0.217389
1,development,soxx,1762,20,0.005963,0.005949,1.002306,-0.079317,-0.168148
2,reused_diagnostic,soxl,915,20,0.115936,0.110643,1.047837,-0.108327,-0.066934
3,reused_diagnostic,soxx,915,20,0.010441,0.009944,1.050030,-0.144507,0.047884


In [4]:
from IPython.display import Markdown, display
display(Markdown((card / "conclusion.md").read_text()))


# H20 匹配期限实验

实际执行 55 次拟合和 8 个原生回放；现金、数量、费用和三个资产权重对账通过。

- development/10bp：收益 325.75%，相对 SOXX 终值 1.0000，MaxDD -47.37%，CVaR5 -4.42%。
- reused_diagnostic/10bp：收益 398.12%，相对 SOXX 终值 1.0000，MaxDD -42.79%，CVaR5 -5.30%。
- development/25bp：收益 325.11%，相对 SOXX 终值 1.0000，MaxDD -47.37%，CVaR5 -4.42%。
- reused_diagnostic/25bp：收益 397.38%，相对 SOXX 终值 1.0000，MaxDD -42.79%，CVaR5 -5.30%。

历史门：False；诊断门：False。不晋升，路线决定：end_h20_route_no_further_search。
这仍是已复用历史；H5 经济参考未核验，不能把差异归因于单独期限。固定 HAC5 不覆盖所有 H20 重叠标签依赖；历史开盘是报价成交代理，并非真实 09:31 成交。
若历史及诊断门通过，需新的冻结身份及至少 60 个独立合格前向日；否则停止 H20，不重搜期限、alpha 或阈值。
Recorder `72455b5609ab4cc5b37808d7e10b2cb0`，FINISHED。


# H20 匹配期限实验

实际执行 55 次拟合和 8 个原生回放；现金、数量、费用和三个资产权重对账通过。

- development/10bp：收益 325.75%，相对 SOXX 终值 1.0000，MaxDD -47.37%，CVaR5 -4.42%。
- reused_diagnostic/10bp：收益 398.12%，相对 SOXX 终值 1.0000，MaxDD -42.79%，CVaR5 -5.30%。
- development/25bp：收益 325.11%，相对 SOXX 终值 1.0000，MaxDD -47.37%，CVaR5 -4.42%。
- reused_diagnostic/25bp：收益 397.38%，相对 SOXX 终值 1.0000，MaxDD -42.79%，CVaR5 -5.30%。

历史门：False；诊断门：False。不晋升，路线决定：end_h20_route_no_further_search。
这仍是已复用历史；H5 经济参考未核验，不能把差异归因于单独期限。固定 HAC5 不覆盖所有 H20 重叠标签依赖；历史开盘是报价成交代理，并非真实 09:31 成交。
若历史及诊断门通过，需新的冻结身份及至少 60 个独立合格前向日；否则停止 H20，不重搜期限、alpha 或阈值。
Recorder `72455b5609ab4cc5b37808d7e10b2cb0`，FINISHED。

补充预测诊断：两个资产、两个窗口的 MSE 均高于各自历史均值 prior；点 rank IC 为负，但本轮未对 IC 计算置信区间，不能据此证明所有信息均无效。见 forecast_information.csv 与分析 Recorder 3556aa4c0f424e4f9d05070869b2e839。
